# Illinois State Building Inventory: Two Altair Visualizations

**Data:** https://raw.githubusercontent.com/UIUC-iSchool-DataViz/is445_data/main/building_inventory.csv

In [1]:
import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

In [2]:
url = "https://raw.githubusercontent.com/UIUC-iSchool-DataViz/is445_data/main/building_inventory.csv"
df = pd.read_csv(url)
print(df.shape)
df.head()

(8862, 22)


,Agency Name,Location Name,Address,City,Zip code,County,Congress Dist,Congressional Full Name,Rep Dist,Rep Full Name,...,Bldg Status,Year Acquired,Year Constructed,Square Footage,Total Floors,Floors Above Grade,Floors Below Grade,Usage Description,Usage Description 2,Usage Description 3
0,Department of Natural Resources,Anderson Lake Conservation Area - Fulton County,Anderson Lake C.a.,Astoria,61501,Fulton,17,Cheri Bustos,93,Hammond Norine K.,...,In Use,1975,1975,144,1,1,0,Unusual,Unusual,Not provided
1,Department of Natural Resources,Anderson Lake Conservation Area - Fulton County,Anderson Lake C.a.,Astoria,61501,Fulton,17,Cheri Bustos,93,Hammond Norine K.,...,In Use,2004,2004,144,1,1,0,Unusual,Unusual,Not provided
2,Department of Natural Resources,Anderson Lake Conservation Area - Fulton County,Anderson Lake C.a.,Astoria,61501,Fulton,17,Cheri Bustos,93,Hammond Norine K.,...,In Use,2004,2004,144,1,1,0,Unusual,Unusual,Not provided
3,Department of Natural Resources,Anderson Lake Conservation Area - Fulton County,Anderson Lake C.a.,Astoria,61501,Fulton,17,Cheri Bustos,93,Hammond Norine K.,...,In Use,2004,2004,144,1,1,0,Unusual,Unusual,Not provided
4,Department of Natural Resources,Anderson Lake Conservation Area - Fulton County,Anderson Lake C.a.,Astoria,61501,Fulton,17,Cheri Bustos,93,Hammond Norine K.,...,In Use,2004,2004,144,1,1,0,Unusual,Unusual,Not provided


In [3]:
print("Year Constructed == 0:", (df["Year Constructed"] == 0).sum())
print("Square Footage == 0:  ", (df["Square Footage"] == 0).sum())
df["Usage Description"].value_counts()

Year Constructed == 0: 289
Square Footage == 0:   57


Usage Description
Storage                 2231
Unusual                 1684
Industrial              1174
Assembly                 948
Detention & Correc       731
Business                 715
Residential              636
Health Care              382
Education                258
Mercantile                77
Utiility & Miscellan      21
Not provided               4
Public                     1
Name: count, dtype: int64

In [ ]:
#cleaning
clean = df[(df["Year Constructed"] > 0) & (df["Square Footage"] > 0)].copy()

clean["Usage"] = clean["Usage Description"].replace({
    "Detention & Correc": "Detention & Correction",
    "Utiility & Miscellan": "Utility & Misc.",
})
rare = ["Mercantile", "Utility & Misc.", "Not provided", "Public"]
clean.loc[clean["Usage"].isin(rare), "Usage"] = "Other"

print(clean.shape)
clean["Usage"].value_counts()

(8548, 23)


Usage
Storage                   2095
Unusual                   1615
Industrial                1142
Assembly                   923
Detention & Correction     726
Business                   694
Residential                628
Health Care                380
Education                  257
Other                       88
Name: count, dtype: int64

In [7]:
def decade_label(year):
    return "Pre-1900" if year < 1900 else f"{(year // 10) * 10}s"

clean["Decade"] = clean["Year Constructed"].apply(decade_label)

by_decade = (
    clean.groupby(["Decade", "Usage"])
         .agg(Buildings=("Square Footage", "size"),
              SqFt=("Square Footage", "sum"))
         .reset_index()
)

decade_order = ["Pre-1900"] + [f"{d}s" for d in range(1900, 2020, 10)]
# order the stack / legend by how common each usage is overall
usage_order = clean["Usage"].value_counts().index.tolist()
# numeric rank so every bar stacks in the same order (most common usage at the bottom)
by_decade["UsageRank"] = by_decade["Usage"].map({u: i for i, u in enumerate(usage_order)})

by_decade.head()

,Decade,Usage,Buildings,SqFt,UsageRank
0,1900s,Assembly,22,910248,3
1,1900s,Business,11,190636,5
2,1900s,Detention & Correction,7,95736,4
3,1900s,Education,5,142919,8
4,1900s,Health Care,1,6400,7


In [8]:
usage_select = alt.selection_point(fields=["Usage"], bind="legend")

decade_chart = alt.Chart(by_decade).mark_bar().encode(
    x=alt.X("Decade:O", sort=decade_order, title="Decade constructed",
            axis=alt.Axis(labelAngle=0)),
    y=alt.Y("sum(Buildings):Q", title="Number of buildings"),
    color=alt.Color("Usage:N", sort=usage_order,
                    scale=alt.Scale(scheme="tableau10"),
                    legend=alt.Legend(title="Usage (click to highlight)")),
    order=alt.Order("UsageRank:Q"),
    opacity=alt.condition(usage_select, alt.value(1.0), alt.value(0.15)),
    tooltip=[
        alt.Tooltip("Decade:O"),
        alt.Tooltip("Usage:N"),
        alt.Tooltip("Buildings:Q", title="Buildings"),
        alt.Tooltip("SqFt:Q", title="Total sq ft", format=",.0f"),
    ],
).add_params(
    usage_select
).properties(
    width="container", height=380,
    title="Illinois state buildings by decade constructed and usage",
)

decade_chart

alt.Chart(...)

In [9]:
decade_chart.save("../assets/json/buildings_by_decade_usage.json")

In [10]:
agency_totals = clean.groupby("Agency Name")["Square Footage"].sum().sort_values(ascending=False)
top_agencies = agency_totals.head(15).index.tolist()
print(f"Top 15 agencies hold {agency_totals.head(15).sum() / agency_totals.sum():.0%} of all square footage")

by_agency_year = (
    clean[clean["Agency Name"].isin(top_agencies)]
         .groupby(["Agency Name", "Year Constructed"])
         .agg(SqFt=("Square Footage", "sum"),
              Buildings=("Square Footage", "size"))
         .reset_index()
         .rename(columns={"Agency Name": "Agency", "Year Constructed": "Year"})
)
print(by_agency_year.shape)
max_sqft = by_agency_year.groupby("Agency")["SqFt"].sum().max()
by_agency_year.head()

Top 15 agencies hold 91% of all square footage
(1042, 4)


,Agency,Year,SqFt,Buildings
0,Department of Agriculture,1894,85000,1
1,Department of Agriculture,1895,19830,3
2,Department of Agriculture,1901,86200,1
3,Department of Agriculture,1903,10288,1
4,Department of Agriculture,1905,8000,1


In [11]:
year_slider = alt.param(
    name="year_built",
    value=2019,
    bind=alt.binding_range(min=1850, max=2019, step=1, name="Built in or before: "),
)

agency_chart = alt.Chart(by_agency_year).transform_filter(
    alt.datum.Year <= year_slider
).transform_aggregate(
    SqFt="sum(SqFt)",
    Buildings="sum(Buildings)",
    groupby=["Agency"],
).transform_calculate(
    AvgSize="datum.SqFt / datum.Buildings"
).mark_bar().encode(
    x=alt.X("SqFt:Q", title="Total square footage",
            scale=alt.Scale(domain=[0, max_sqft]),
            axis=alt.Axis(format="~s")),
    y=alt.Y("Agency:N", sort="-x", title=None,
            axis=alt.Axis(labelLimit=260)),
    color=alt.Color("AvgSize:Q", title="Avg sq ft per building",
                    scale=alt.Scale(type="log", scheme="viridis",
                                    domain=[500, 100000], clamp=True),
                    legend=alt.Legend(format="~s")),
    tooltip=[
        alt.Tooltip("Agency:N"),
        alt.Tooltip("SqFt:Q", title="Total sq ft", format=",.0f"),
        alt.Tooltip("Buildings:Q", title="Buildings"),
        alt.Tooltip("AvgSize:Q", title="Avg sq ft / building", format=",.0f"),
    ],
).add_params(
    year_slider
).properties(
    width="container", height=420,
    title="Floor space of the 15 largest state agencies, by year built",
)

agency_chart

alt.Chart(...)

In [12]:
agency_chart.save("../assets/json/agency_sqft_slider.json")